# 신용카드 채무불이행 고객 예측

https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset

In [ ]:
# !pip install xlrd --break-system-packages

In [ ]:
import pandas as pd
import urllib.request
import os

os.makedirs('./data', exist_ok=True)

# UCI 원본
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls"
urllib.request.urlretrieve(url, './data/UCI_Credit_Card.xls')

# xls 읽기
df = pd.read_excel('./data/UCI_Credit_Card.xls', header=1)
df.to_csv('./data/UCI_Credit_Card.csv', index=False)
print(df.shape)

# df = pd.read_csv('./data/UCI_Credit_Card.csv')
card_df = df.drop('ID', axis=1)
card_df.head(3)

In [ ]:
card_df = card_df.rename(columns={'default payment next month': 'default'})

y_target = card_df['default']
X_features = card_df.drop('default', axis=1)

<프롬프트>

채무불이행 예측 모델 의 target은 deault 변수, 이후 모델링 프로세스를 정리해 보세요. 내가 정리결과를 검토하고 피드백을 제공하겠습니다. 한단계씩 진행하겠습니다. 코드를 한단계씩만 제공하세요

모델링 프로세스 요약

Target: default (0 = 정상, 1 = 채무불이행)

1단계 EDA
- 타깃 클래스 비율 확인 (불균형 여부)
- 결측치/이상치 탐색
- 주요 피처 분포 및 타깃과의 상관관계 시각화

2단계 데이터 전처리
- 범주형 피처 인코딩 (필요 시)
- 수치형 피처 스케일링 (StandardScaler 등)
- 불필요 컬럼 제거

3단계 Train/Test Split
- train_test_split에서 stratify=y 옵션으로 클래스 비율 유지
- 일반적으로 80:20 분할

4단계 모델 학습
- Logistic Regression 
- Random Forest (베이스라인)
- XGBoost

5단계 모델 평가
- Accuracy, Precision, Recall, F1-Score
- ROC-AUC Curve (불균형 데이터에서 중요)

6단계 피처 중요도
- 트리 기반 모델 feature importance 시각화
- 예측에 영향을 주는 핵심 변수 파악

7단계 하이퍼파라미터 튜닝 (선택)
- GridSearchCV or RandomizedSearchCV
- 최적 파라미터로 최종 모델 재학습

# EDA

>프롬프트>
위의 전체 프로세스를 노트북에 마크다운 셀을 추가하고 쓰세요. 다만 마크다운의 h2,h3, --- 같은 태그는 배제합니다.

In [ ]:
# 타깃 클래스 비율 확인
print("=== 타깃 클래스 분포 ===")
print(y_target.value_counts())
print()
print(f"정상(0)     : {y_target.value_counts()[0]:,}명  ({y_target.value_counts(normalize=True)[0]:.1%})")
print(f"채무불이행(1): {y_target.value_counts()[1]:,}명  ({y_target.value_counts(normalize=True)[1]:.1%})")

분석 결과

- 정상(0): 23,364명 (77.9%)
- 채무불이행(1): 6,636명 (22.1%)
- 비율 약 4:1 -> 클래스 불균형(Class Imbalance) 존재

모델링 시 주의사항
- Accuracy만으로는 평가가 부족함 (전부 0으로 예측해도 77.9%)
- Recall, F1-Score, ROC-AUC를 주요 평가지표로 사용
- 채무불이행(소수 클래스) 미탐지 비용이 커서 Recall이 특히 중요
- 필요 시 class_weight='balanced' or 오버샘플링(SMOTE) 적용

# EDA 2단계: 결측치, 중복, 이상치, 범주 분포 점검

목적
- 모델링 전에 데이터 품질을 점검합니다.
- 이후 평가는 Recall, F1-Score, ROC-AUC 중심으로 진행합니다.
- 클래스 불균형 대응 순서는 class_weight='balanced' 적용 후 SMOTE 적용입니다.

In [ ]:
import numpy as np
import pandas as pd

print('=== 결측치 개수 (상위 10개) ===')
missing = card_df.isnull().sum().sort_values(ascending=False)
print(missing.head(10))
print(f'총 결측치 수: {int(missing.sum()):,}')

print('\n=== 중복 행 개수 ===')
dup_count = card_df.duplicated().sum()
print(f'중복 행 수: {dup_count:,}')

print('\n=== 수치형 변수 기술통계 (소수점 4자리) ===')
desc = card_df.describe().T.astype(float)
display(desc.style.format('{:,.4f}'))

# IQR 기준 이상치 비율 계산
num_cols = card_df.columns.drop('default')
q1 = card_df[num_cols].quantile(0.25)
q3 = card_df[num_cols].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outlier_ratio = ((card_df[num_cols] < lower) | (card_df[num_cols] > upper)).mean().sort_values(ascending=False)

print('\n=== 이상치 비율 상위 10개 (IQR 기준, %) ===')
outlier_top10 = (outlier_ratio.head(10) * 100).to_frame('이상치_비율(%)').astype(float)
display(outlier_top10.style.format('{:,.4f}'))

print('\n=== 주요 범주형(코드형) 변수 분포 ===')
for col in ['SEX', 'EDUCATION', 'MARRIAGE']:
    print(f'\n[{col}]')
    print(card_df[col].value_counts().sort_index())


EDA 2단계 요약

- 결측치, 중복, 이상치, 범주 분포를 확인한 뒤 학습/검증 분할을 진행합니다.
- 모델 비교는 Recall, F1-Score, ROC-AUC를 중심으로 수행합니다.
- 불균형 대응은 다음 순서로 실험합니다.
  1. class_weight='balanced'
  2. SMOTE 오버샘플링

## 기술통계 분석 (신용카드 채무불이행 데이터)

### 1. 데이터 개요
- 관측치: **30,000건**
- 변수 수: **24개** (`ID` 제외, 타깃 `default` 포함)
- 타깃 비율:
  - 정상(0): **77.88%**
  - 채무불이행(1): **22.12%**
- 해석: 클래스 불균형이 존재하므로 정확도(Accuracy) 단독 평가는 부적절하며, **Recall / F1-Score / ROC-AUC** 중심 평가가 필요함.

### 2. 핵심 수치형 변수 요약
- `LIMIT_BAL` (신용한도)
  - 평균: **167,484.32**
  - 중앙값: **140,000**
  - 표준편차: **129,747.66**
  - 최소~최대: **10,000 ~ 1,000,000**
  - 해석: 평균이 중앙값보다 커 **우측 꼬리 분포(고한도 소수 고객)** 가능성이 큼.

- `AGE` (나이)
  - 평균: **35.49**
  - 중앙값: **34**
  - 표준편차: **9.22**
  - 최소~최대: **21 ~ 79**
  - 해석: 30~40대 중심 분포로 보이며 극단적 연령은 상대적으로 적음.

### 3. 청구금액/상환금액 패턴
- `BILL_AMT1~6` 평균은 최근월에서 과거월로 갈수록 대\ccb4로 감소
  - 예: `BILL_AMT1` **51,223.33** -> `BILL_AMT6` **38,871.76**
- `PAY_AMT1~6` 평균은 약 **4,800 ~ 5,900** 수준
  - 월별 평균 변동은 있으나 전반적으로 비슷한 범위 유지
- 해석: 청구금액 규모 대비 실제 상환금액은 상대적으로 작아, 연체 위험군 탐지 신호로 활용 가능함.

### 4. 범주형(코드형) 변수 분포
- `SEX`
  - 1: 11,888명 (**39.63%**)
  - 2: 18,112명 (**60.37%**)

- `EDUCATION`
  - 1: 10,585명 (**35.28%**)
  - 2: 14,030명 (**46.77%**)
  - 3: 4,917명 (**16.39%**)
  - 0/4/5/6: 합계 468명 (**1.56%**)
  - 해석: 소수 코드(0,4,5,6)는 모델링 시 병합 여부 검토 필요.

- `MARRIAGE`
  - 1: 13,659명 (**45.53%**)
  - 2: 15,964명 (**53.21%**)
  - 3: 323명 (**1.08%**)
  - 0: 54명 (**0.18%**)
  - 해석: 희소 범주(0,3)는 전처리 정책이 필요.

### 5. 이상치(IQR 기준) 점검
- 이상치 비율 상위 변수(%)
  - `PAY_2` **14.70**
  - `PAY_3` **14.03**
  - `PAY_4` **11.69**
  - `PAY_0` **10.43**
  - `PAY_6` **10.26**
- 해석: 연체 관련 `PAY_*` 변수에서 이상치 비율이 높아, 단순 제거보다 로버스트 전략을 우선 검토하는 것이 적절함.

### 6. 모델링 시사점
- 불균형 데이터이므로 평가는 **Recall, F1-Score, ROC-AUC** 중심으로 수행함.
- 실험 순서
  1. `class_weight='balanced'` 적용 모델
  2. `SMOTE` 오버샘플링 적용 모델
- 동일 데이터 분할과 동일 평가지표로 비교하여 최종 모델을 선정함.

# EDA 3단계: 시각화

- 타깃 분포 확인
- 주요 수치형 변수 분포 및 타깃별 비교
- 범주형 변수별 채무불이행률 비교
- 상관관계 히트맵 확인

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

# 1) Target distribution
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=card_df, x='default', palette='Set2', ax=ax)
ax.set_title('Target Distribution (default)')
ax.set_xlabel('default (0: normal, 1: default)')
ax.set_ylabel('count')
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

# 2) Numeric distributions by target
num_cols_for_plot = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols_for_plot):
    sns.histplot(data=card_df, x=col, hue='default', bins=40, kde=True,
                 stat='density', common_norm=False, alpha=0.35, ax=axes[i])
    axes[i].set_title(f'{col} Distribution by Target')
plt.tight_layout()
plt.show()

# 3) Boxplots for outlier check
box_cols = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for i, col in enumerate(box_cols):
    sns.boxplot(data=card_df, x='default', y=col, palette='Set3', ax=axes[i])
    axes[i].set_title(f'{col} by Target')
plt.tight_layout()
plt.show()

# 4) Default rate by categorical features
cat_cols = ['SEX', 'EDUCATION', 'MARRIAGE']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, col in enumerate(cat_cols):
    rate = card_df.groupby(col)['default'].mean().reset_index()
    sns.barplot(data=rate, x=col, y='default', color='#5B8FF9', ax=axes[i])
    axes[i].set_title(f'Default Rate by {col}')
    axes[i].set_ylabel('default rate')
    axes[i].set_ylim(0, max(0.35, rate['default'].max() * 1.15))
plt.tight_layout()
plt.show()

# 5) Correlation heatmap (top 15 with target)
corr = card_df.corr(numeric_only=True)
top_cols = corr['default'].abs().sort_values(ascending=False).head(15).index
heat_df = card_df[top_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(heat_df, cmap='coolwarm', center=0, annot=False, square=False)
plt.title('Correlation Heatmap (Top 15 features by |corr with default|)')
plt.tight_layout()
plt.show()

## EDA 4단계: 전체 변수 상관관계 상세 분석

- 전체 수치형 변수 히트맵(상관계수 숫자 표시)
- `default`와의 상관계수 정렬 표
- 변수쌍 간 상관계수 상위 분석

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

numeric_df = card_df.select_dtypes(include=[np.number]).copy()
corr_all = numeric_df.corr(numeric_only=True)

print(f'전체 수치형 변수 개수: {corr_all.shape[0]}')
display(corr_all.round(3))

# 1) 전체 상관관계 히트맵 (상관계수 숫자 표시)
plt.figure(figsize=(18, 14))
sns.heatmap(corr_all, cmap='coolwarm', center=0, vmin=-1, vmax=1, annot=True, fmt='.2f', annot_kws={'size': 7}, linewidths=0.3, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap (All Numeric Variables)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# 2) 타깃(default)과의 상관관계 정렬 표
corr_with_target = corr_all['default'].drop('default').sort_values(key=lambda s: s.abs(), ascending=False)
corr_with_target_df = corr_with_target.to_frame('corr_with_default')
corr_with_target_df['abs_corr'] = corr_with_target_df['corr_with_default'].abs()
print('\n=== default와의 상관관계(|corr| 기준 정렬) ===')
display(corr_with_target_df.round(4))

# 3) 변수쌍 간 상관관계 상위(중복 제거)
upper = corr_all.where(np.triu(np.ones(corr_all.shape), k=1).astype(bool))
pairs = upper.stack().reset_index().rename(columns={'level_0':'var1','level_1':'var2',0:'corr'})
pairs['abs_corr'] = pairs['corr'].abs()
pairs_sorted = pairs.sort_values('abs_corr', ascending=False).reset_index(drop=True)
print('\n=== 변수쌍 상관관계 상위 20개(|corr| 기준) ===')
display(pairs_sorted.head(20).round(4))

# 4) 강한 상관관계(기본 임계값 |corr| >= 0.7)
strong_pairs = pairs_sorted[pairs_sorted['abs_corr'] >= 0.7]
print(f'\n강한 상관관계 변수쌍 개수(|corr| >= 0.7): {len(strong_pairs)}')
display(strong_pairs.round(4))

# 5) default와의 상관 Top10 막대그래프
top10 = corr_with_target.head(10).sort_values()
plt.figure(figsize=(8, 6))
plt.barh(top10.index, top10.values, color=['#d95f02' if v > 0 else '#1b9e77' for v in top10.values])
plt.axvline(0, color='black', linewidth=1)
plt.title('Top 10 Correlations with default')
plt.xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()


## EDA 차트 분석

### 1. 타깃 분포 차트 (Countplot)
- `default=0`이 `default=1`보다 뚰렷하게 많음.
- 채무불이행 비율이 약 22% 수준으로, 클래스 불균형이 존재함.
- 해석: 모델 평가는 Accuracy보다 **Recall, F1-Score, ROC-AUC** 중심이 적절함.

### 2. 수치형 변수 분포 차트 (Histplot + KDE)
- `LIMIT_BAL`:
  - 낮은~중간 한도 구간에 관측치가 밀집되고, 고한도 구간으로 갈수록 빈도가 감소함.
  - 우측 꼬리가 긴 분포 특성이 보임.
- `AGE`:
  - 30~40대 중심의 단봉형 분포.
  - 극단 연령은 상대적으로 적음.
- `BILL_AMT1`, `PAY_AMT1`:
  - 일부 큰 금액 구간까지 넓게 분포하며 왜도(치우침)가 존재함.
- 해석: 로그 변환/스케일링 여부에 따라 모델 안정성이 개선될 수 있음.

### 3. 타깃별 박스플롯 (Boxplot)
- `default=1` 집단이 일부 변수에서 중앙값/분산/이상치 패턴이 `default=0`과 다르게 나타남.
- 특히 청구·상환 관련 변수에서 이상치가 다수 관측됨.
- 해석: 단순 이상치 제거보다, 모델 강건성(트리 기반, class_weight, SMOTE) 활용이 유리함.

### 4. 범주형 변수별 채무불이행률 (Barplot)
- `SEX`, `EDUCATION`, `MARRIAGE` 코드별로 default rate 차이가 존재함.
- 일부 희소 범주(표본 수가 적은 코드)는 변동성이 커 보일 수 있음.
- 해석: 희소 범주는 병합 검토가 필요하며, 범주별 리스크 차이는 유의미한 신호가 될 수 있음.

### 5. 상관 히트맵 (타깃 기준 상위 변수)
- 연체 이력(`PAY_*`) 계열이 타깃과 상대적으로 높은 상관을 보일 가능성이 큼.
- 청구금액(`BILL_AMT*`)과 상환금액(`PAY_AMT*`)은 서로 군집된 상관 구조를 형성함.
- 해석: 선형모델은 규제화(regularization), 트리모델은 중요도 해석을 병행하는 것이 적절함.

## 종합 결론
- 데이터는 **클래스 불균형 + 왜도/이상치 + 범주형 리스크 차이**가 동시에 존재함.
- 다음 단계는 계획한 대로
  1. `class_weight='balanced'` 적용 모델
  2. `SMOTE` 적용 모델
  을 동일 조건에서 비교하고, **Recall/F1/ROC-AUC** 기준으로 최종 모델을 선정하는 것이 타당함.

# 2단계 데이터 전처리

- 범주형 피처 인코딩 (OneHotEncoder)
- 수치형 피처 스케일링 (StandardScaler)
- 불필요 컬럼 제거

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

target_col = 'default'

# 1) 불필요 컬럼 제거 (존재할 때만 제거)
drop_candidates = ['ID']
drop_cols = [c for c in drop_candidates if c in card_df.columns]

X = card_df.drop(columns=[target_col] + drop_cols).copy()
y = card_df[target_col].copy()

# 2) 범주형/수치형 컬럼 분리
categorical_cols = [c for c in ['SEX', 'EDUCATION', 'MARRIAGE'] if c in X.columns]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

print('제거된 컬럼:', drop_cols if drop_cols else '없음')
print('범주형 컬럼:', categorical_cols)
print('수치형 컬럼 수:', len(numeric_cols))

# 3) 학습/테스트 분할 (클래스 비율 유지)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4) 전처리기: 범주형 인코딩 + 수치형 스케일링
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ]
)

X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

print('\n전처리 완료')
print('X_train 원본 shape:', X_train.shape)
print('X_test 원본 shape :', X_test.shape)
print('X_train 전처리 shape:', X_train_preprocessed.shape)
print('X_test 전처리 shape :', X_test_preprocessed.shape)

# 이후 단계에서 재사용할 변수명 정리
X_train_processed = X_train_preprocessed
X_test_processed = X_test_preprocessed


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 베이스라인 랜덤포레스트
rf_baseline = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_baseline.fit(X_train_processed, y_train)

y_pred = rf_baseline.predict(X_test_processed)
y_proba = rf_baseline.predict_proba(X_test_processed)[:, 1]

recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print('=== RandomForest Baseline Metrics ===')
print(f'Recall : {recall:.4f}')
print(f'F1-Score: {f1:.4f}')
print(f'ROC-AUC : {roc_auc:.4f}')

print('\n=== Classification Report ===')
print(classification_report(y_test, y_pred, digits=4))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - RandomForest Baseline')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

baseline_result = {
    'model': 'RandomForest Baseline',
    'recall': float(recall),
    'f1': float(f1),
    'roc_auc': float(roc_auc),
}
baseline_result


## 모델 결과 분석 (RandomForest Baseline)

### 1) 핵심 성능 지표
- Recall: **0.3677**
- F1-Score: **0.4652**
- ROC-AUC: **0.7579**
- Accuracy: **0.8130**

### 2) 성능 해석
- ROC-AUC가 0.75 수준이므로 전반적인 순위 구분 능력은 나쁘지 않지만, 운영 관점에서 핵심인 Recall이 낮음.
- 채무불이행(1) 클래스 Recall 0.3677은 실제 불이행 고객의 약 36.8%만 찾아냈다는 의미로, 미탐지(FN) 비율이 큰 편임.
- Accuracy는 0.8130으로 높게 보이지만, 클래스 불균형(0클래스 다수) 혁향을 감안하면 단독 평가 지표로는 부적절함.

### 3) 분류 리포트 관점 요약
- 정상(0) 클래스는 Precision/Recall이 모두 높아 잘 맞추고 있음.
- 불이행(1) 클래스는 Precision(0.6329)대비 Recall(0.3677)이 낮아, 양성 탐지보다 보수적으로 예측하는 패턴을 보임.

### 4) 다음 개선 방향
1. `class_weight='balanced'` 적용 모델로 Recall 개선 확인
2. SMOTE 오버샘플링 적용 후 동일 지표(Recall/F1/ROC-AUC) 비교
3. 필요 시 판별 임계값(Threshold) 조정으로 Recall 우선 최적화

## 베이스라인 결과 기록

- 위 셀 실행 후 `baseline_result` 값을 아래에 기록하세요.
- 다음 단계에서 `class_weight='balanced'`, `SMOTE` 모델과 동일 지표(Recall/F1/ROC-AUC)로 비교합니다.

## 비교 1: RandomForest (`class_weight='balanced'`)

- 목표: 소수 클래스(채무불이행=1) 탐지률 Recall 개선
- 평가지표: Recall, F1-Score, ROC-AUC

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

rf_balanced = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_balanced.fit(X_train_processed, y_train)

y_pred_bal = rf_balanced.predict(X_test_processed)
y_proba_bal = rf_balanced.predict_proba(X_test_processed)[:, 1]

recall_bal = recall_score(y_test, y_pred_bal)
f1_bal = f1_score(y_test, y_pred_bal)
roc_auc_bal = roc_auc_score(y_test, y_proba_bal)

print('=== RandomForest (class_weight=balanced) ===')
print(f'Recall : {recall_bal:.4f}')
print(f'F1-Score: {f1_bal:.4f}')
print(f'ROC-AUC : {roc_auc_bal:.4f}')

print('\n=== Classification Report ===')
print(classification_report(y_test, y_pred_bal, digits=4))

cm_bal = confusion_matrix(y_test, y_pred_bal)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_bal, annot=True, fmt='d', cmap='Greens')
plt.title('Confusion Matrix - RF class_weight=balanced')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

result_balanced = {
    'model': 'RF class_weight=balanced',
    'recall': float(recall_bal),
    'f1': float(f1_bal),
    'roc_auc': float(roc_auc_bal)
}
result_balanced


## 비교 2: RandomForest + SMOTE 오버샘플링

- 목표: 학습 데이터에서 소수 클래스를 오버샘플링하여 불균형 완화
- 평가지표: Recall, F1-Score, ROC-AUC

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)

print('SMOTE 적용 전 클래스 분포:')
print(y_train.value_counts())
print('\nSMOTE 적용 후 클래스 분포:')
print(y_train_smote.value_counts())

rf_smote = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_smote.fit(X_train_smote, y_train_smote)

y_pred_sm = rf_smote.predict(X_test_processed)
y_proba_sm = rf_smote.predict_proba(X_test_processed)[:, 1]

recall_sm = recall_score(y_test, y_pred_sm)
f1_sm = f1_score(y_test, y_pred_sm)
roc_auc_sm = roc_auc_score(y_test, y_proba_sm)

print('=== RandomForest + SMOTE ===')
print(f'Recall : {recall_sm:.4f}')
print(f'F1-Score: {f1_sm:.4f}')
print(f'ROC-AUC : {roc_auc_sm:.4f}')

print('\n=== Classification Report ===')
print(classification_report(y_test, y_pred_sm, digits=4))

cm_sm = confusion_matrix(y_test, y_pred_sm)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_sm, annot=True, fmt='d', cmap='Oranges')
plt.title('Confusion Matrix - RF + SMOTE')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

result_smote = {
    'model': 'RF + SMOTE',
    'recall': float(recall_sm),
    'f1': float(f1_sm),
    'roc_auc': float(roc_auc_sm)
}
result_smote


## 비교 요약 표


In [ ]:
import pandas as pd

required = ['baseline_result', 'result_balanced', 'result_smote']
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f'다음 결과 변수가 없습니다: {missing}. 먼저 각 모델 셀을 실행하세요.')

summary_rows = [
    {
        '모델': 'RandomForest 기본',
        'Recall': baseline_result['recall'],
        'F1-Score': baseline_result['f1'],
        'ROC-AUC': baseline_result['roc_auc'],
    },
    {
        '모델': "RandomForest (class_weight='balanced')",
        'Recall': result_balanced['recall'],
        'F1-Score': result_balanced['f1'],
        'ROC-AUC': result_balanced['roc_auc'],
    },
    {
        '모델': 'RandomForest + SMOTE',
        'Recall': result_smote['recall'],
        'F1-Score': result_smote['f1'],
        'ROC-AUC': result_smote['roc_auc'],
    },
]

compare_df = pd.DataFrame(summary_rows)
display(compare_df.sort_values(by='Recall', ascending=False).reset_index(drop=True).style.format({'Recall':'{:.4f}','F1-Score':'{:.4f}','ROC-AUC':'{:.4f}'}))


## 3개 모델 비교 결과 (채무불이행(1) 기준)

| 모델 | Precision | Recall | F1-Score | ROC-AUC |
|---|---:|---:|---:|---:|
| 베이스라인 RFC | 0.64 | 0.36 | 0.46 | 0.7572 |
| RFC + class_weight='balanced' | 0.64 | 0.34 | 0.45 | 0.7583 |
| RFC + SMOTE | 0.51 | 0.48 | 0.49 | 0.7443 |

## 결과 해석

- SMOTE 적용 시 Recall이 **0.36 -> 0.48**로 가장 큰 폭으로 개선됨 (**+0.12**)
- 단, Precision은 **0.64 -> 0.51**로 하락하여, 정상 고객을 채무불이행으로 잘못 분류하는 비율(FP)이 증가함
- ROC-AUC는 세 모델 모두 **0.74 ~ 0.76** 수준으로 큰 차이 없음
- 채무불이행 탐지(Recall) 관점에서는 **SMOTE 모델이 가장 우수**함

## 최종 선택

- 채무불이행 탐지가 목적인 비즈니스 관점에서 **RFC + SMOTE를 최종 모델로 선택**함